## 04 — Regression Modeling

**Objective:** Predict each player's `rating_title` (FotMob match rating) from their 5-match rolling performance features. Trained on seasons 2021/22–2023/24, evaluated on 2024/25.

### Outfield Players
1. Load scaled train/test parquets and define feature/target columns
2. Establish mean predictor baseline (MSE, RMSE, R²)
3. Fit Ridge regression with TimeSeriesSplit cross-validation to tune `alpha`
4. Fit Random Forest regressor with TimeSeriesSplit cross-validation to tune `n_estimators` and `max_depth`
5. Evaluate both models on test set — MSE, RMSE, R²
6. Compare baseline vs Ridge vs Random Forest in summary table
7. Plot Ridge coefficients — features with strongest positive/negative effect on rating
8. Plot Random Forest feature importances — most informative features
9. Plot predicted vs actual rating_title (residual analysis)
10. Check residuals by position group for systematic bias

### Goalkeepers
11. Repeat steps 2–9 for GK using GK-specific features
12. Compare outfield vs GK model performance

### Export
13. Save Ridge and Random Forest models to `data/models/`
14. Save train and test predictions as new columns in parquets to `data/processed/datasets/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('data/processed/datasets')

outfield_train = pd.read_parquet(PROCESSED_DIR / 'outfield_train_scaled.parquet')
outfield_test  = pd.read_parquet(PROCESSED_DIR / 'outfield_test_scaled.parquet')
gk_train       = pd.read_parquet(PROCESSED_DIR / 'gk_train_scaled.parquet')
gk_test        = pd.read_parquet(PROCESSED_DIR / 'gk_test_scaled.parquet')

ID_COLS = [
    'match_id', 'round', 'match_date', 'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name', 'minutes_played',
    'shirt_number', 'position_id', 'is_goalkeeper', 'season',
    'home_away', 'opponent', 'team_goals_scored', 'opp_goals_scored', 'result',
    'position_group'
]

print(f'Outfield train : {outfield_train.shape[0]:,} rows x {outfield_train.shape[1]} cols')
print(f'Outfield test  : {outfield_test.shape[0]:,} rows x {outfield_test.shape[1]} cols')
print(f'GK train       : {gk_train.shape[0]:,} rows x {gk_train.shape[1]} cols')
print(f'GK test        : {gk_test.shape[0]:,} rows x {gk_test.shape[1]} cols')

# Define feature and target columns
FEATURE_COLS_OUTFIELD = [c for c in outfield_train.columns if c not in ID_COLS + ['rating_title']]
FEATURE_COLS_GK       = [c for c in gk_train.columns if c not in ID_COLS + ['rating_title']]

print(f'\nOutfield features : {len(FEATURE_COLS_OUTFIELD)}')
print(f'GK features       : {len(FEATURE_COLS_GK)}')

Outfield train : 21,847 rows x 57 cols
Outfield test  : 7,034 rows x 57 cols
GK train       : 2,193 rows x 47 cols
GK test        : 713 rows x 47 cols

Outfield features : 36
GK features       : 26
